# 10 — MURA all-anatomy data preparation

This local development extension covers all seven MURA anatomy categories. It uses **official MURA train only** and creates a single internal patient-level train/validation split. Official MURA `valid` is deliberately excluded.

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

def locate_project_root():
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "CHARTER.md").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the repository checkout.")

ROOT = locate_project_root()
RUNTIME = ROOT.parent / 'runtime' / 'multianatomy_v1'
MANIFESTS = RUNTIME / 'manifests'
QC = RUNTIME / 'qc'
BASELINE = RUNTIME / 'baseline_all_v1'
assert ROOT.name == 'mura-wrist-ml-pipeline', ROOT
print('Repository: local project checkout')
print('Runtime artifacts: local only (not committed to Git)')


Repository: local project checkout
Runtime artifacts: local only (not committed to Git)


In [2]:
summary = pd.read_csv(MANIFESTS / 'development_split_summary.csv')
provenance = json.loads((MANIFESTS / 'development_provenance.json').read_text(encoding='utf-8'))
assert provenance['official_valid_read'] is False
assert provenance['test_used_for_selection'] is False
display(Markdown('**Data-governance check passed:** official MURA `valid` was not read.'))
display(summary)

**Data-governance check passed:** official MURA `valid` was not read.

,anatomy,final_split,patients,studies,images,normal_studies,abnormal_studies,mean_views_per_study
0,XR_ELBOW,train,1377,1409,3962,879,530,2.812
1,XR_ELBOW,val,334,345,969,215,130,2.809
2,XR_FINGER,train,1495,1543,4060,1017,526,2.631
3,XR_FINGER,val,370,392,1046,263,129,2.668
4,XR_FOREARM,train,683,694,1437,462,232,2.071
5,XR_FOREARM,val,182,183,388,128,55,2.120
6,XR_HAND,train,1551,1607,4413,1193,414,2.746
7,XR_HAND,val,394,411,1130,304,107,2.749
8,XR_HUMERUS,train,471,474,1019,258,216,2.150
9,XR_HUMERUS,val,116,118,253,63,55,2.144


In [3]:
pivot = summary.pivot(index='anatomy', columns='final_split', values='studies').fillna(0)
ax = pivot.plot(kind='bar', figsize=(10, 4.5), color=['#2563eb', '#0f766e'])
ax.set(title='Study counts by anatomy and internal split', xlabel='Anatomy', ylabel='Studies')
ax.grid(axis='y', alpha=0.25)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

C:\Users\netaf\AppData\Local\Temp\ipykernel_17896\1220459730.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Important design choice

The same patient can appear in more than one body part. Therefore the split is global by patient — not a separate split per anatomy. Each study ID includes the anatomy (`XR_*`) so images from different body parts cannot be grouped into one study.

In [4]:
combined = pd.read_csv(MANIFESTS / 'all_anatomies_train_manifest.csv')
study_audit = combined.groupby('study_id').agg(
    anatomy=('anatomy', 'nunique'), patient=('patient_id', 'nunique'),
    label=('label', 'nunique'), declared_views=('num_views_in_study', 'first'),
    indexed_views=('image_path', 'nunique')
).reset_index()
assert study_audit['anatomy'].eq(1).all()
assert study_audit['patient'].eq(1).all()
assert study_audit['label'].eq(1).all()
assert study_audit['declared_views'].eq(study_audit['indexed_views']).all()
print({'studies': len(study_audit), 'images': len(combined), 'unique_patients': combined['patient_id'].nunique()})
print('Study hierarchy and view-count checks passed.')

{'studies': 13457, 'images': 36808, 'unique_patients': 11184}
Study hierarchy and view-count checks passed.
